In [ ]:
# Code annotation/comments for this can be found in the lab_2.ipynb
import sys
!{sys.executable} -m pip install cryptography

import socket
import os
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
import pickle

with open("public_key.pem", "rb") as f:
    public_key = serialization.load_pem_public_key(f.read())
message = b"Secure message from sender: this data is confidential."
aes_key = os.urandom(32) 
iv = os.urandom(16)

cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
encryptor = cipher.encryptor()
encrypted_message = encryptor.update(message) + encryptor.finalize()
encrypted_key = public_key.encrypt(
aes_key,
padding.OAEP(
mgf=padding.MGF1(algorithm=hashes.SHA256()),
algorithm=hashes.SHA256(),
label=None
)
)
payload = pickle.dumps((encrypted_key, iv, encrypted_message))
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect(("localhost", 65432))
    s.sendall(payload)
    print("Message encrypted and sent successfully.")

✅ Encrypted message sent!



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# Modules and libaries used
import socket
import os
import pickle

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
# Reads public key
with open("public_key.pem", "rb") as f:
    receiver_public_key = serialization.load_pem_public_key(f.read())
# Reads private key
with open("private_key.pem", "rb") as f:
    sender_private_key = serialization.load_pem_private_key(f.read(), password=None)
# Has a message to be sent
message = b"hello world"
# Creates AES keys 
aes_key = os.urandom(32)  
iv = os.urandom(16)
# Creates variables to use to encrypt the message
cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
encryptor = cipher.encryptor()
encrypted_message = encryptor.update(message) + encryptor.finalize()

# Encrypts the AES key using public key
encrypted_key = receiver_public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Creates signature using private key and hashes it
signature = sender_private_key.sign(
    iv + encrypted_message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)
# Bundles up the data and sends it
payload = pickle.dumps((encrypted_key, iv, encrypted_message, signature))

# TCP connection open
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect(("localhost", 65432))
    s.sendall(payload)
    print("message sent")


message sent


In [ ]:
# Same as code above but with a minor change, as we are sneding a message that has been 
# tampered with in order to test the receiver.
import socket
import os
import pickle

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

with open("public_key.pem", "rb") as f:
    receiver_public_key = serialization.load_pem_public_key(f.read())

with open("private_key.pem", "rb") as f:
    sender_private_key = serialization.load_pem_private_key(f.read(), password=None)

message = b"hello world"

aes_key = os.urandom(32) 
iv = os.urandom(16)

cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv))
encryptor = cipher.encryptor()
encrypted_message = encryptor.update(message) + encryptor.finalize()

encrypted_key = receiver_public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

signature = sender_private_key.sign(
    iv + encrypted_message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)
# Flips a byte to change the message around
encrypted_message = bytearray(encrypted_message)
encrypted_message[0] ^= 0xFF 
encrypted_message = bytes(encrypted_message)

payload = pickle.dumps((encrypted_key, iv, encrypted_message, signature))

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect(("localhost", 65432))
    s.sendall(payload)
    print("message sent")


message sent
